# SRA Taxonomy Analysis Tool (STAT)

This notebook demonstrates how to use [NCBI STAT](https://www.ncbi.nlm.nih.gov/sra/docs/sra-taxonomy-analysis-tool/) to generate taxonomy-derived information from a small sequence input.

We will also include a short concept demo showing what human read scrubbing means: identifying reads that may be human-derived and either masking or removing them from an output FASTQ file.

## What is STAT?

The SRA Taxonomy Analysis Tool, or STAT, is a collection of tools for building and querying k-mer databases. It compares sequencing reads against taxonomically labeled k-mer databases and reports taxonomy-derived information about the reads in a sequencing run.

In practical terms, STAT can help answer questions like:

- What organisms appear to be represented in this sequence data?
- Does the sample contain bacteria, viruses, fungi, parasites, or human-associated sequence?
- Are there unexpected taxonomic signals in the data?
- Could this dataset be useful for further analysis?

STAT is useful for quickly exploring the taxonomic content of sequence data. It uses exact k-mer matches against precomputed dictionary databases, with workflows that can include a broad first pass to identify likely organisms and a more focused second pass to refine taxonomic assignments.

## What about human read scrubbing?

STAT is related to human read scrubbing workflows such as NCBI’s Human Read Removal Tool, also known as HRRT or `sra-human-scrubber`.

Human read scrubbing is different from general taxonomy reporting. Instead of summarizing all organisms detected in a sample, the goal is to identify reads that may be human-derived and produce a cleaned output file.

In a typical scrubbing workflow, flagged reads may be:

- **Masked**, where the sequence is replaced with `N` characters
- **Removed**, where the flagged read is omitted from the output FASTQ

This notebook does **not** run the full production human scrubber. Instead, it includes a small teaching example that demonstrates the difference between masking and removing reads.

## Commands and tools used in this demo

| Command, script, or section | What it does in this demo |
|---|---|
| `quickbuild.sh` | Builds the minimal STAT tools needed to run the bundled example. |
| `build_index_of_each_file` | Builds per-file k-mer indexes from the example FASTA database files. |
| `merge_db` | Merges the per-file indexes into one combined example database. |
| `identify_tax_ids` | Links database k-mers to taxonomy IDs using the `tax.parents` file. |
| `db_tax_id_to_dbs` | Combines the database and taxonomy-ID mapping into a `.dbs` file. |
| `sort_dbs` | Sorts the dense `.dbs` file by tax ID for optional two-step workflows. |
| `aligns_to` | Queries the input FASTA file against the STAT database and writes the `.hits` output file. |
| `sra-human-scrubber` | Masks and removes flagged reads. |

**Note:** Commands that begin with `!` run shell commands inside the notebook. Cells that begin with `%%bash` run the whole cell as Bash.

---

## 1. Set up the workspace

In this code block, we will downloads the `tax` branch of `ncbi/ngs-tools` into `~/stat_demo_workspace/ngs-tools-stat` and store the STAT tool path in `STAT_DIR`.

Using a fixed workspace avoids the rerun bug where the repo was cloned inside a previous `tools/tax` folder.

In [ ]:
# Clone STAT into a fixed workspace so rerunning the notebook does not nest repos.

from pathlib import Path
import os
import shutil

REPO_URL = "https://github.com/ncbi/ngs-tools.git"
REPO_BRANCH = "tax"
DEMO_ROOT = Path.home() / "stat_demo_workspace"
REPO_DIR = DEMO_ROOT / "ngs-tools-stat"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

DEMO_ROOT.mkdir(parents=True, exist_ok=True)

!git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

STAT_DIR = REPO_DIR / "tools" / "tax"
os.environ["STAT_DIR"] = str(STAT_DIR)

print("STAT directory:", STAT_DIR)

print("\n\n-------------\nTASK COMPLETE\n-------------")

## 2. Review STAT quickstart file

Inspect the README, bundled example FASTA inputs, and the key commands in the official STAT example script before running anything.

**NOTE:** This isn't necessary to set up an analysis, but it should be a good place to start when it comes time to integrate STAT into your specific pipeline.

In [ ]:
# Inspect the STAT quick start and the bundled example.

!sed -n '1,40p' "$STAT_DIR/README.md"

print("\n--- Example input files ---")
!find "$STAT_DIR/examples/example_data" -maxdepth 1 -type f | sort | sed -n '1,20p'

print("\n--- Key commands in the official example ---")
!grep -nEi 'DENSE_WINDOW|SPARSE_WINDOW|KMER_LEN|build_index|merge_db|identify_tax_ids|db_tax_id_to_dbs|sort_dbs|aligns_to|hits_to_report|python2' "$STAT_DIR/examples/build_db_and_run.sh" | sed -n '1,100p'

print("\n\n-------------\nTASK COMPLETE\n-------------")

## 3. Build the minimal STAT tools

Next we will run STAT's `quickbuild.sh` minimal build script and confirm that the command-line tools have been installed.

If the build somehow fails, then you should check that your current environment has Git, Python 3, a C++ compiler, and OpenMP support.

In [ ]:
%%bash
# Build the minimal STAT tools needed for the example.

echo "Running STAT's minimal build script."
echo "Please be patient. This might take a moment..."

set -euo pipefail
cd "$STAT_DIR"

bash ./quickbuild.sh

echo
echo "--- Tools used in this demo ---"
for tool in aligns_to build_index_of_each_file merge_db identify_tax_ids db_tax_id_to_dbs sort_dbs; do
    if [ -x "./bin/$tool" ]; then
        echo "./bin/$tool"
    elif [ -e "./bin/$tool" ]; then
        echo "./bin/$tool found, but not marked executable"
    else
        echo "./bin/$tool not found"
    fi
done

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"

## 4. Align a FASTA to a k-mer database

In this section, we will first build a small demo k-mer database from the bundled example reference files, and then use the `aligns_to` command to query one bundled FASTA file against that database.

`aligns_to` is one of the main STAT query commands you will want to use. It compares reads from a FASTA (in this case, `SRR4841604.fasta`) against a STAT database, writing the matching taxonomy-derived hits to a`*.hits` file (here, it is `SRR4841604.fasta.hits`).

The notebook stops after this first query so the demo stays short and avoids later optional reporting steps. For additional workflows, including two-step processing and report generation, see the official [STAT GitHub repository](https://github.com/ncbi/ngs-tools/tree/tax/tools/tax).

In [ ]:
%%bash
# Run a shortened version of the official STAT example.
# Builds the small example database and runs the first aligns_to query only.

set -euo pipefail
cd "$STAT_DIR/examples"

# Clean old demo outputs if this notebook is rerun.
rm -f files.list.example example.* SRR4841604.fasta*.hits *.log build_db_and_run_demo.sh tax_list taxdump.tar.gz
find sequence_tree \( -name '*.dense.db' -o -name '*.sparse.db' -o -name '*.summary.tsv' \) -type f -exec rm -f {} +

# Copy the official example, through to the first one-step aligns_to command.
awk '{ print } /\$bin_dir\/aligns_to -dbs .*SRR4841604\.fasta/ { exit }' build_db_and_run.sh > build_db_and_run_demo.sh

if ! grep -q 'aligns_to' build_db_and_run_demo.sh; then
    echo "Could not find the one-step aligns_to command in build_db_and_run.sh."
    exit 1
fi

echo "--- STAT query command used in this demo ---"
grep -n 'aligns_to' build_db_and_run_demo.sh

if ! bash ./build_db_and_run_demo.sh > stat_demo_run.log 2>&1; then
    echo
    echo "--- Demo run failed; last 80 log lines ---"
    tail -80 stat_demo_run.log
    exit 1
fi

echo
echo "--- Selected run log lines ---"
grep -Ei 'version|window size|kmer len|kmers|FastaReader|total spot count|total read count|total time' stat_demo_run.log | sed -n '1,80p'

test -s SRR4841604.fasta.hits

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"

## 5. Preview the taxonomy-derived output

At this point, we should have successfully created our `.hits` output file.

Each row starts with an input read/spot identifier followed by a taxon-hit summary such as `211044x17`; in this example, `211044` is a taxonomic identifier and `17` is the associated hit count for that record.

**NOTE:** In this shortened demo, the `.hits` file is the main output we will inspect. In a fuller STAT workflow, this `.hits` file would usually be treated as an intermediate result that can be summarized into a more readable taxonomy report.

In [ ]:
%%bash
# Preview only useful text outputs from the demo.

set -euo pipefail
cd "$STAT_DIR/examples"

echo "--- Main STAT output files ---"
ls -lh SRR4841604.fasta.hits stat_demo_run.log

echo
echo "--- First lines of the STAT hit output ---"
sed -n '1,12p' SRR4841604.fasta.hits

echo
echo "--- A few index summary tables created during database build ---"
find sequence_tree -name '*.summary.tsv' -type f | sort | head -n 3 | while read -r f; do
    echo
    echo "===== $f ====="
    sed -n '1,5p' "$f"
done

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"


## 6. (Optional) Create a full STAT report

As previously described, this demo stops at creating a `.hits` file because the official reporting helpers in this STAT example expect a `python2` command to be available. This requires additional steps which would complicate this quickstart Jupyter demo.

That said, if you want to set up a `python2` environment for your own workflow, and have it installed and available.

In [ ]:
%%bash
# (Optional) Activate Python 2 environment.
# Uncomment the below two lines if you installed Python 2 in a conda environment.

#source "$(conda info --base)/etc/profile.d/conda.sh"
#conda activate stat-py2

set -euo pipefail

cd "$STAT_DIR/examples"

bin_dir="../bin"

# Step 1: Run a broad first pass against the sparse database.
$bin_dir/aligns_to -dbs ./example.sparse.dbs ./example_data/SRR4841604.fasta > ./SRR4841604.fasta.1ststep.hits

# Step 2: Convert first-pass hits into a tax ID list.
python2 $bin_dir/hits_to_tax_list.py ./SRR4841604.fasta.1ststep.hits > ./tax_list

# Step 3: Run a focused second pass against the dense database,
# using only the tax IDs found in the first pass.
$bin_dir/aligns_to -dbss ./example.dense.dbss -tax_list ./tax_list ./example_data/SRR4841604.fasta > ./SRR4841604.fasta.2ndstep.hits

# Step 4: Generate a readable report from the second-pass hits.
$bin_dir/hits_to_report.sh ./SRR4841604.fasta.2ndstep.hits

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"

⚠️ **IMPORTANT!** If you try to run the above codeblock in a notebook that (likely) doesn't have Python2 installed, it will fail. This is expected!

## 7. Human read scrubbing with sra-human-scrubber

STAT is related to NCBI’s Human Read Removal Tool, also called HRRT or `sra-human-scrubber`.

The full HRRT workflow uses a human-focused k-mer database to identify reads that may be human-derived. By default, identified reads are **masked** by replacing the sequence with `N` characters. The tool can also be run in a mode where identified reads are **removed** from the output FASTQ entirely.

In this notebook environment, we will not run the full production scrubber. Instead, this section uses a small demo FASTQ file and a small demo list of “human-like” marker k-mers to demonstrate the two main behaviors:

- **Masking** flagged reads with `N`
- **Removing** flagged reads from the output file

> This is a teaching example only. The marker k-mers below are artificial and should not be used for real human-read removal.

In [ ]:
from pathlib import Path

# Create a tiny teaching FASTQ.
# These are artificial DNA sequences. The marker k-mers below are NOT real human markers.
records = [
    ("@read_001_bacterial_like", "GATCGATCGATCGATCGATC"),
    ("@read_002_demo_human_like", "TTTTACGTACGTACGTAAAA"),
    ("@read_003_viral_like", "CCCAACCAACCAACCGGGT"),
    ("@read_004_demo_human_like", "GGGTTTAAACCCGGGTTTAA"),
]

input_fastq = Path("demo_input.fastq")
masked_fastq = Path("demo_output.masked.fastq")
removed_fastq = Path("demo_output.removed.fastq")
flagged_spots = Path("demo_flagged_spots.txt")


def write_demo_fastq(path, records):
    """Write a tiny FASTQ file using high-quality placeholder scores."""
    with path.open("w") as out:
        for header, seq in records:
            out.write(f"{header}\n{seq}\n+\n{'I' * len(seq)}\n")


write_demo_fastq(input_fastq, records)

# Demo marker k-mers used only for demonstration.
# A real scrubber uses a curated database, not a two-item list like this.
demo_human_marker_kmers = {
    "ACGTACGTACGT",
    "GGGTTTAAA",
}


def read_fastq(path):
    """Simple FASTQ parser for this small demo."""
    with path.open() as handle:
        while True:
            header = handle.readline().rstrip("\n")
            if not header:
                break

            seq = handle.readline().rstrip("\n")
            plus = handle.readline().rstrip("\n")
            qual = handle.readline().rstrip("\n")

            if not header.startswith("@") or plus != "+":
                raise ValueError(f"{path} does not look like a valid FASTQ file near {header!r}")

            if len(seq) != len(qual):
                raise ValueError(f"Sequence and quality lengths do not match for {header}")

            yield header, seq, plus, qual


def scrub_fastq(input_path, output_path, marker_kmers, mode="mask", flagged_path=None):
    """
    Demo scrubber.

    mode="mask":
        Keep flagged reads, but replace their sequence with Ns.

    mode="remove":
        Omit flagged reads entirely from the output FASTQ.
    """
    if mode not in {"mask", "remove"}:
        raise ValueError("mode must be 'mask' or 'remove'")

    total = 0
    flagged = []
    written = 0

    with output_path.open("w") as out_handle:
        for header, seq, plus, qual in read_fastq(input_path):
            total += 1
            is_flagged = any(kmer in seq for kmer in marker_kmers)

            if is_flagged:
                flagged.append(header[1:].split()[0])

                if mode == "remove":
                    continue

                if mode == "mask":
                    seq = "N" * len(seq)

            out_handle.write(f"{header}\n{seq}\n{plus}\n{qual}\n")
            written += 1

    if flagged_path is not None:
        flagged_path.write_text("\n".join(flagged) + ("\n" if flagged else ""))

    return {
        "mode": mode,
        "input_records": total,
        "flagged_records": len(flagged),
        "output_records": written,
        "flagged_read_ids": flagged,
    }


masked_summary = scrub_fastq(
    input_fastq,
    masked_fastq,
    demo_human_marker_kmers,
    mode="mask",
    flagged_path=flagged_spots,
)

removed_summary = scrub_fastq(
    input_fastq,
    removed_fastq,
    demo_human_marker_kmers,
    mode="remove",
)

print("--- Scrubbing summaries ---")
print(masked_summary)
print(removed_summary)

print("\n--- Flagged read IDs ---")
print(flagged_spots.read_text())

print("--- Original FASTQ ---")
print(input_fastq.read_text())

print("--- Masked FASTQ: flagged reads are preserved, but sequence is replaced with Ns ---")
print(masked_fastq.read_text())

print("--- Removed FASTQ: flagged reads are omitted entirely ---")
print(removed_fastq.read_text())

print("\n\n-------------\nTASK COMPLETE\n-------------")

### What happened?

The scrubber used a human-focused k-mer database to identify reads/spots that may be human-derived.

In the masked output, flagged sequence content is replaced with `N` characters. This keeps the FASTQ record structure but hides the sequence.

In the removed output, flagged spots are removed from the output FASTQ entirely.

For real submissions or production workflows, use the scrubber on your own FASTQ files and follow your institution’s data-sharing, consent, and privacy requirements.

---

If you have any more specific questions, please visit the [STAT webpage](https://www.ncbi.nlm.nih.gov/sra/docs/sra-taxonomy-analysis-tool/), the [STAT GitHub repository](https://github.com/ncbi/ngs-tools/tree/tax/tools/tax), or email your questions to [sra@ncbi.nlm.nih.gov](mailto:sra@ncbi.nlm.nih.gov).
